In [19]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torch.optim import Adam
from PIL import Image

from pathlib import Path


from torch.cuda.amp import autocast, GradScaler




In [20]:
class mydataset(Dataset):
    def __init__(self):
        self.data_sketch=[]
        self.data_photo=[]


        parent_dir = '/kaggle/input/datasets/arhamfsd/sketch-photos/genai_ass3_q3_reduced/256x256/photo/tx_000000000000'
        image_extensions = ('.png', '.jpg', '.jpeg', '.bmp')
        
        parent_path = Path(parent_dir)
        
        
        
        for folder in sorted(parent_path.iterdir()):
            if folder.is_dir():
                for file in sorted(folder.rglob("*")):
                    if file.suffix.lower() in image_extensions:
                        self.data_photo.append(str(file))
        
        size=len(self.data_photo)
        
        paren_dir='/kaggle/input/datasets/arhamfsd/sketch-photos/genai_ass3_q3_reduced/256x256/sketch/tx_000000000010'
        

        parent_path = Path(parent_dir)
        
        self.temp=[]
        
        for folder in (parent_path.iterdir()):
            if folder.is_dir():
                for file in (folder.rglob("*")):
                    if file.suffix.lower() in image_extensions:
                        self.temp.append(str(file))

        print(size)
        

        self.data_sketch=self.temp[:]
        print(len(self.data_sketch))
        

    def __len__(self):
        return len(self.data_sketch)

    def __getitem__(self, idx):
        photo = Image.open(self.data_photo[idx]).convert('RGB')
        photo = photo.resize((128, 128))
        sketch= Image.open(self.data_sketch[idx]).convert('RGB')
        sketch = sketch.resize((128, 128))
        return photo, sketch    


In [21]:
traindata=  mydataset()
train_data=DataLoader(traindata,batch_size=64,shuffle=True,num_workers=2,pin_memory=True)

12500
12500


In [22]:
class resnet(nn.Module):
    def __init__(self):
        super().__init__()

        self.convulation=nn.Conv2d(3, 32, kernel_size=7, stride=2, padding=3, bias=False)
        self.batchnormalization=nn.BatchNorm2d(32)
        self.relu=nn.ReLU(inplace=True)

        self.convulation1=nn.Conv2d(32, 32, kernel_size=3, stride=1, padding=1, bias=False)
        self.batchnormalization1=nn.BatchNorm2d(32)
    
        

    def forward(self, x):
        original_x=x
        x=self.convulation(x)
        x=self.batchnormalization(x)
        x=self.relu(x)
        x=self.convulation1(x)
        x=self.batchnormalization1(x)
        x+=original_x
        x=self.relu(x)
        return x

In [23]:
class sketch_generator(nn.Module):
    def __init__(self):
        super().__init__()
        self.generator = nn.Sequential(

            nn.ReflectionPad2d(3),
            nn.Conv2d(3, 64, kernel_size=7, stride=1),
            nn.InstanceNorm2d(64),
            nn.ReLU(inplace=True),

            
            nn.Conv2d(64, 128, 3, 2, 1),
            nn.InstanceNorm2d(128),
            nn.ReLU(inplace=True),

            
            nn.Conv2d(128, 256, 3, 2, 1),
            nn.InstanceNorm2d(256),
            nn.ReLU(inplace=True),

            
            resnet(),
            resnet(),
            resnet(),
            resnet(),
            resnet(),
            resnet(),

            
            nn.ConvTranspose2d(256, 128, 3, 2, 1, output_padding=1),
            nn.InstanceNorm2d(128),
            nn.ReLU(inplace=True),

            
            nn.ConvTranspose2d(128, 64, 3, 2, 1, output_padding=1),
            nn.InstanceNorm2d(64),
            nn.ReLU(inplace=True),

            
            nn.ReflectionPad2d(3),
            nn.Conv2d(64, 3, kernel_size=7, stride=1),
            nn.Tanh()
    )


    def forward(self, x):
        return self.generator(x)

In [24]:
class sketch_discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.discriminator=nn.Sequential(
            nn.Conv2d(in_channels=3,out_channels=64,kernel_size=4,stride=2,padding=1),
            nn.LeakyReLU(0.2,inplace=True),

            nn.Conv2d(in_channels=64,out_channels=128,kernel_size=4,stride=2,padding=1),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2,inplace=True),

            nn.Conv2d(in_channels=128,out_channels=256,kernel_size=4,stride=2,padding=1),
            nn.BatchNorm2d (256),
            nn.LeakyReLU(0.2,inplace=True),

            nn.Conv2d(in_channels=256,out_channels=512,kernel_size=4,stride=2,padding=1),
            nn.BatchNorm2d(512),
            nn.LeakyReLU(0.2,inplace=True ),

            nn.Conv2d(in_channels=512,out_channels=1,kernel_size=4,stride=1,padding=0),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.discriminator(x)

In [25]:
class photo_generator(nn.Module):
    def __init__(self):
        super().__init__()
        self.generator = nn.Sequential(

            nn.ReflectionPad2d(3),
            nn.Conv2d(3, 64, kernel_size=7, stride=1),
            nn.InstanceNorm2d(64),
            nn.ReLU(inplace=True),

            
            nn.Conv2d(64, 128, 3, 2, 1),
            nn.InstanceNorm2d(128),
            nn.ReLU(inplace=True),

            
            nn.Conv2d(128, 256, 3, 2, 1),
            nn.InstanceNorm2d(256),
            nn.ReLU(inplace=True),

            
            resnet(),
            resnet(),
            resnet(),
            resnet(),
            resnet(),
            resnet(),

            
            nn.ConvTranspose2d(256, 128, 3, 2, 1, output_padding=1),
            nn.InstanceNorm2d(128),
            nn.ReLU(inplace=True),

            
            nn.ConvTranspose2d(128, 64, 3, 2, 1, output_padding=1),
            nn.InstanceNorm2d(64),
            nn.ReLU(inplace=True),

            
            nn.ReflectionPad2d(3),
            nn.Conv2d(64, 3, kernel_size=7, stride=1),
            nn.Tanh()
    )


    def forward(self, x):
        return self.generator(x)

In [26]:
class photo_discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(

            
            nn.Conv2d(3, 64, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2, inplace=True),

    
            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1),
            nn.InstanceNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),

            
            nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1),
            nn.InstanceNorm2d(256),
            nn.LeakyReLU(0.2, inplace=True),

            
            nn.Conv2d(256, 512, kernel_size=4, stride=1, padding=1),  # NOTE: stride=1 here
            nn.InstanceNorm2d(512),
            nn.LeakyReLU(0.2, inplace=True),

            
            nn.Conv2d(512, 1, kernel_size=4, stride=1, padding=1)
        )


    def forward(self, x):
        return self.discriminator(x)

In [27]:
photo_discriminator=photo_discriminator()
sketch_discriminator=sketch_discriminator() 
photo_generator=photo_generator()
sketch_generator=sketch_generator()

device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')

photo_generator=torch.nn.DataParallel(photo_generator)
sketch_generator=torch.nn.DataParallel(sketch_generator)
photo_discriminator=torch.nn.DataParallel(photo_discriminator)
sketch_discriminator=torch.nn.DataParallel(sketch_discriminator)

photo_discriminator.to(device)
sketch_discriminator.to(device)
photo_generator.to(device)
sketch_generator.to(device)

DataParallel(
  (module): sketch_generator(
    (generator): Sequential(
      (0): ReflectionPad2d((3, 3, 3, 3))
      (1): Conv2d(3, 64, kernel_size=(7, 7), stride=(1, 1))
      (2): InstanceNorm2d(64, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
      (3): ReLU(inplace=True)
      (4): Conv2d(64, 128, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
      (5): InstanceNorm2d(128, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
      (6): ReLU(inplace=True)
      (7): Conv2d(128, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
      (8): InstanceNorm2d(256, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
      (9): ReLU(inplace=True)
      (10): resnet(
        (convulation): Conv2d(3, 32, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
        (batchnormalization): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (convulation1): 

In [28]:
epochs=20
learning_rate=0.0002

photo_generator_optim=Adam(photo_generator.parameters(),lr=learning_rate,betas=(0.5,0.999))
photo_discriminator_optim=Adam(photo_discriminator.parameters(),lr=learning_rate,betas=(0.5,0.999))
sketch_generator_optim=Adam(sketch_generator.parameters(),lr=learning_rate,betas=(0.5,0.999))
sketch_discriminator_optim=Adam(sketch_discriminator.parameters(),lr=learning_rate,betas=(0.5,0.999))

adversarial_loss = nn.MSELoss()  
cycle_loss = nn.L1Loss()

In [ ]:
photo_gen_error=[]
photo_dis_error=[]
sketch_gen_error=[]
sketch_dis_error=[]

In [ ]:
for epoch in range(epochs):
    
    for photo,sketch in train_data:

        photo=photo.to(device)
        sketch=sketch.to(device)
        
        photo_discriminator.zero_grad()
        sketch_discriminator.zero_grad()
        
        outputsketch=sketch_generator(photo).view(-1)
        outputphoto=photo_generator(sketch).view(-1)

        recon_photo = photo_generator(outputsketch)
        recon_sketch = sketch_generator(outputphoto)

        valid = torch.ones((photo.size(0), 1, 14, 14), device=device)
        fake = torch.zeros((photo.size(0), 1, 14, 14), device=device)

        loss_sketch_dis=
        loss_photo_dis=
        
        loss_sketch_dis.backward()
        loss_photo_dis.backward()

        sketch_discriminator_optim.step()
        photo_discriminator_optim.step()



############################################################################
        photo_generator.zero_grad()
        sketch_generator.zero_grad()

        loss_sketch_gen=
        loss_photo_gen=

        
        loss_sketch_gen.backward()
        loss_photo_gen.backward()
        photo_generator_optim.step()
        sketch_generator_optim.step()

        photo_gen_error.append(loss_photo_gen.item())
        photo_dis_error.append(loss_photo_dis.item())
        sketch_gen_error.append(loss_sketch_gen.item())
        sketch_dis_error.append(loss_sketch_dis.item())
        




In [ ]:
torch.save(generatorGan.state_dict(),'generatorWGan2.pth')
torch.save(discriminatorGan.state_dict(),'criticWGan1.pth')